### Supervisor 에이전트 시스템의 구조
어떤 작업을 직접 수행하는게 아닌, 어떤 에이전트를 골라야할지 직접 판단하는 역할이다.
1. suvervisor 에이전트 시스템에서 하나의 supervisor 에이전트가 작업 수행을 위해 하위 에이전트 명령을 하달한다.
2. 하위 에이전트들은 작업 명령을 받으면, 수행 결과를 supervisor에 전달하고, 이결과를 반복한다.

- Hierarchical(계층적) 구조도 이와같은 형식으로 동작한다.


- 복잡한 작업에대해서, supervisor에게 자율성을 부여하고, 관리를한다. supervisor가 판단하기에 노드호출하고, 사용자의 질문을 처리한다.

- 여러개의 agent를 효과적으로 관리하는 방법이다.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

### PDF기반 retrieveal chain 생성

In [ ]:
### yfinance

In [5]:
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
from langchain.tools import tool
ticker = yf.Ticker("AAPL")

historical_prices = ticker.history(period="5d", interval="1d")
last_5_days_close = historical_prices["Close"].tail(5) # 종가정보
last_5_days_close_dict = {date.strftime("%Y-%m-%d"): price for date, price in last_5_days_close.items()} # 딕셔너리 형태로 가져온다
last_5_days_close_dict

{'2025-04-07': 181.4600067138672,
 '2025-04-08': 172.4199981689453,
 '2025-04-09': 198.85000610351562,
 '2025-04-10': 190.4199981689453,
 '2025-04-11': 198.14999389648438}

In [3]:
import yfinance as yf
import pandas as pd
import json
from langchain.tools import tool

@tool
def stock_analysis(ticker: str) -> str:
    """
    주어진 주식 티커에 대한 업데이트된 종합적인 재무 분석을 수행합니다.
    """
    def format_number(number):
        if number is None or pd.isna(number):
            return "N/A"
        return f"{number:,.2f}"

    def calculate_key_metrics():
        # 현재와 작년의 데이터 가져오기 (정렬된 순서 확인)
        revenue_data = financials.loc["Total Revenue"].dropna()
        current_revenue = revenue_data.iloc[0]  # 최신 연도 데이터
        previous_revenue = revenue_data.iloc[1]  # 이전 연도 데이터

        net_income_data = financials.loc["Net Income"].dropna()
        current_net_income = net_income_data.iloc[0]
        previous_net_income = net_income_data.iloc[1]

        eps_data = financials.loc["Diluted EPS"].dropna()
        current_eps = eps_data.iloc[0]
        previous_eps = eps_data.iloc[1]

        current_operating_income = financials.loc["Operating Income"].dropna().iloc[0]
        current_stockholders_equity = balance_sheet.loc["Stockholders Equity"].dropna().iloc[0]
        current_total_assets = balance_sheet.loc["Total Assets"].dropna().iloc[0]
        current_total_liabilities = balance_sheet.loc["Total Liabilities Net Minority Interest"].dropna().iloc[0]

        # 성장성 (올해와 작년 기준)
        revenue_growth = ((current_revenue - previous_revenue) / previous_revenue) * 100 if previous_revenue else None
        net_income_growth = ((current_net_income - previous_net_income) / previous_net_income) * 100 if previous_net_income else None
        eps_growth = ((current_eps - previous_eps) / previous_eps) * 100 if previous_eps else None

        # 수익성
        operating_margin = (current_operating_income / current_revenue * 100) if current_revenue else None
        net_profit_margin = (current_net_income / current_revenue * 100) if current_revenue else None
        roe = (current_net_income / current_stockholders_equity * 100) if current_stockholders_equity else None
        roa = (current_net_income / current_total_assets * 100) if current_total_assets else None

        # 밸류에이션
        per = (historical_prices['Close'].iloc[-1] / current_eps) if current_eps else None
        book_value_per_share = (current_stockholders_equity / ticker.info.get("sharesOutstanding", 1)) if current_stockholders_equity else None
        pbr = (historical_prices['Close'].iloc[-1] / book_value_per_share) if book_value_per_share else None

        # 배당
        dividend_yield = ticker.info.get("dividendYield", 0) * 100
        dividend_payout_ratio = (ticker.info.get("dividendRate", 0) / current_eps * 100) if current_eps else None

        # 리스크
        debt_to_equity = (current_total_liabilities / current_stockholders_equity * 100) if current_stockholders_equity else None

        return {
            "성장성": {
                "매출 성장률": format_number(revenue_growth) + "%" if revenue_growth else "N/A",
                "순이익 성장률": format_number(net_income_growth) + "%" if net_income_growth else "N/A",
                "EPS 성장률": format_number(eps_growth) + "%" if eps_growth else "N/A",
            },
            "수익성": {
                "영업이익률": format_number(operating_margin) + "%" if operating_margin else "N/A",
                "순이익률": format_number(net_profit_margin) + "%" if net_profit_margin else "N/A",
                "ROE": format_number(roe) + "%" if roe else "N/A",
                "ROA": format_number(roa) + "%" if roa else "N/A",
            },
            "밸류에이션": {
                "PER": format_number(per) if per else "N/A",
                "PBR": format_number(pbr) if pbr else "N/A",
            },
            "배당": {
                "배당 수익률": format_number(dividend_yield) + "%" if dividend_yield else "N/A",
                "배당 성향": format_number(dividend_payout_ratio) + "%" if dividend_payout_ratio else "N/A",
            },
            "리스크": {
                "부채비율": format_number(debt_to_equity) + "%" if debt_to_equity else "N/A",
            }
        }

    ticker = yf.Ticker(ticker)
    historical_prices = ticker.history(period="1y")
    last_5_days_close = historical_prices['Close'].tail(5)
    last_5_days_close_dict = {date.strftime('%Y-%m-%d'): price for date, price in last_5_days_close.items()}
    financials = ticker.financials
    balance_sheet = ticker.balance_sheet
    key_metrics = calculate_key_metrics()

    result = {
        "최근 5일간 종가": last_5_days_close_dict,
        **key_metrics
    }

    return json.dumps(result, indent=4, ensure_ascii=False)

# 실행 예제
print(stock_analysis("AAPL"))


{
    "최근 5일간 종가": {
        "2024-12-12": 247.9600067138672,
        "2024-12-13": 248.1300048828125,
        "2024-12-16": 251.0399932861328,
        "2024-12-17": 253.47999572753906,
        "2024-12-18": 248.0500030517578
    },
    "성장성": {
        "매출 성장률": "2.02%",
        "순이익 성장률": "-3.36%",
        "EPS 성장률": "0.33%"
    },
    "수익성": {
        "영업이익률": "31.51%",
        "순이익률": "23.97%",
        "ROE": "164.59%",
        "ROA": "25.68%"
    },
    "밸류에이션": {
        "PER": "40.46",
        "PBR": "65.84"
    },
    "배당": {
        "배당 수익률": "0.40%",
        "배당 성향": "16.31%"
    },
    "리스크": {
        "부채비율": "540.88%"
    }
}


In [ ]:
import yfinance as yf
import pandas as pd
import json
from langchain.tools import tool

@tool
def stock_analysis(ticker: str) -> str:
    """
    주어진 주식 Ticker에 대한 업데이트된 종합적인 재무 분석을 수행합니다.
    """
    def format_number(number):
        if number is None or pd.isna(number):
            return "N/A"
        return f"{number:,.2f}"

    def calculate_key_metrics():
        # 현재와 작년의 데이터 가져오기 (정렬된 순서 확인)
        revenue_data = financials.loc["Total Revenue"].dropna()
        current_revenue = revenue_data.iloc[0]  # 최신 연도 데이터
        previous_revenue = revenue_data.iloc[1]  # 이전 연도 데이터

        net_income_data = financials.loc["Net Income"].dropna()
        current_net_income = net_income_data.iloc[0]
        previous_net_income = net_income_data.iloc[1]

        eps_data = financials.loc["Diluted EPS"].dropna()
        current_eps = eps_data.iloc[0]
        previous_eps = eps_data.iloc[1]

        current_operating_income = financials.loc["Operating Income"].dropna().iloc[0]
        current_stockholders_equity = balance_sheet.loc["Stockholders Equity"].dropna().iloc[0]
        current_total_assets = balance_sheet.loc["Total Assets"].dropna().iloc[0]
        current_total_liabilities = balance_sheet.loc["Total Liabilities Net Minority Interest"].dropna().iloc[0]

        # 성장성 (올해와 작년 기준)
        revenue_growth = ((current_revenue - previous_revenue) / previous_revenue) * 100 if previous_revenue else None
        net_income_growth = ((current_net_income - previous_net_income) / previous_net_income) * 100 if previous_net_income else None
        eps_growth = ((current_eps - previous_eps) / previous_eps) * 100 if previous_eps else None

        # 수익성
        operating_margin = (current_operating_income / current_revenue * 100) if current_revenue else None
        net_profit_margin = (current_net_income / current_revenue * 100) if current_revenue else None
        roe = (current_net_income / current_stockholders_equity * 100) if current_stockholders_equity else None
        roa = (current_net_income / current_total_assets * 100) if current_total_assets else None

        # 밸류에이션
        per = (historical_prices['Close'].iloc[-1] / current_eps) if current_eps else None
        book_value_per_share = (current_stockholders_equity / ticker.info.get("sharesOutstanding", 1)) if current_stockholders_equity else None
        pbr = (historical_prices['Close'].iloc[-1] / book_value_per_share) if book_value_per_share else None

        # 배당
        dividend_yield = ticker.info.get("dividendYield", 0) * 100
        dividend_payout_ratio = (ticker.info.get("dividendRate", 0) / current_eps * 100) if current_eps else None

        # 리스크
        debt_to_equity = (current_total_liabilities / current_stockholders_equity * 100) if current_stockholders_equity else None

        return {
            "성장성": {
                "매출 성장률": format_number(revenue_growth) + "%" if revenue_growth else "N/A",
                "순이익 성장률": format_number(net_income_growth) + "%" if net_income_growth else "N/A",
                "EPS 성장률": format_number(eps_growth) + "%" if eps_growth else "N/A",
            },
            "수익성": {
                "영업이익률": format_number(operating_margin) + "%" if operating_margin else "N/A",
                "순이익률": format_number(net_profit_margin) + "%" if net_profit_margin else "N/A",
                "ROE": format_number(roe) + "%" if roe else "N/A",
                "ROA": format_number(roa) + "%" if roa else "N/A",
            },
            "밸류에이션": {
                "PER": format_number(per) if per else "N/A",
                "PBR": format_number(pbr) if pbr else "N/A",
            },
            "배당": {
                "배당 수익률": format_number(dividend_yield) + "%" if dividend_yield else "N/A",
                "배당 성향": format_number(dividend_payout_ratio) + "%" if dividend_payout_ratio else "N/A",
            },
            "리스크": {
                "부채비율": format_number(debt_to_equity) + "%" if debt_to_equity else "N/A",
            }
        }

    def format_financial_summary(financials):
        summary = {}
        for date in financials.columns:
            summary[date.strftime("%Y-%m-%d")] = {
                "총수익": format_number(financials.loc['Total Revenue', date]) if 'Total Revenue' in financials.index else "N/A",
                "영업이익": format_number(financials.loc['Operating Income', date]) if 'Operating Income' in financials.index else "N/A",
                "순이익": format_number(financials.loc['Net Income', date]) if 'Net Income' in financials.index else "N/A",
                "EPS": format_number(financials.loc['Diluted EPS', date]) if 'Diluted EPS' in financials.index else "N/A",
            }
        return summary

    def format_balance_sheet_summary(balance_sheet):
        summary = {}
        for date in balance_sheet.columns:
            summary[date.strftime("%Y-%m-%d")] = {
                "총자산": format_number(balance_sheet.loc['Total Assets', date]) if 'Total Assets' in balance_sheet.index else "N/A",
                "총부채": format_number(balance_sheet.loc['Total Liabilities Net Minority Interest', date]) if 'Total Liabilities Net Minority Interest' in balance_sheet.index else "N/A",
                "주주자본": format_number(balance_sheet.loc['Stockholders Equity', date]) if 'Stockholders Equity' in balance_sheet.index else "N/A",
            }
        return summary

    ticker = yf.Ticker(ticker)
    historical_prices = ticker.history(period="1y")
    last_5_days_close = historical_prices['Close'].tail(5)
    last_5_days_close_dict = {date.strftime('%Y-%m-%d'): f"{price:.3f}" for date, price in last_5_days_close.items()}
    financials = ticker.financials
    balance_sheet = ticker.balance_sheet
    key_metrics = calculate_key_metrics()

    result = {
        "최근 5일간 종가": last_5_days_close_dict,
        "재무제표 요약": format_financial_summary(financials),
        "연간 대차대조표": format_balance_sheet_summary(balance_sheet),
        **key_metrics
    }

    return json.dumps(result, indent=4, ensure_ascii=False)

# 실행 예제
print(stock_analysis("AAPL"))


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_29604\4260109940.py:122: LangChainDeprecationWarning: The method `BaseTool.__call__` was deprecated in langchain-core 0.1.47 and will be removed in 1.0. Use :meth:`~invoke` instead.
  print(stock_analysis("AAPL"))
Failed to multipart ingest runs: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=d7c8b5a3-a727-4912-9d31-cb2723ff608a,id=d7c8b5a3-a727-4912-9d31-cb2723ff608a


{
    "최근 5일간 종가": {
        "2025-04-07": "181.460",
        "2025-04-08": "172.420",
        "2025-04-09": "198.850",
        "2025-04-10": "190.420",
        "2025-04-11": "198.150"
    },
    "재무제표 요약": {
        "2024-09-30": {
            "총수익": "391,035,000,000.00",
            "영업이익": "123,216,000,000.00",
            "순이익": "93,736,000,000.00",
            "EPS": "6.08"
        },
        "2023-09-30": {
            "총수익": "383,285,000,000.00",
            "영업이익": "114,301,000,000.00",
            "순이익": "96,995,000,000.00",
            "EPS": "6.13"
        },
        "2022-09-30": {
            "총수익": "394,328,000,000.00",
            "영업이익": "119,437,000,000.00",
            "순이익": "99,803,000,000.00",
            "EPS": "6.11"
        },
        "2021-09-30": {
            "총수익": "365,817,000,000.00",
            "영업이익": "108,949,000,000.00",
            "순이익": "94,680,000,000.00",
            "EPS": "5.61"
        },
        "2020-09-30": {
            "총수익": "N/A",
     

Failed to multipart ingest runs: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=d7c8b5a3-a727-4912-9d31-cb2723ff608a,id=d7c8b5a3-a727-4912-9d31-cb2723ff608a
Failed to multipart ingest runs: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=9b8e9deb-70de-47ab-a56e-3375dc390e08,id=9b8e9deb-70de-47ab-a56e-3375dc390e08; trace=9b8e9deb-70de-47ab-a56e-3375dc390e08,id=cd88da06-ecf3-4d8a-915e-495a0a4b7fc4
Failed to multipart ingest runs:

In [7]:
from langchain_openai import ChatOpenAI

@tool
def investment_decision(ticker: str) -> str:
    """
    LLM을 사용하여 주식의 투자 가치를 평가합니다.
    """
    # 분석 데이터를 가져오기
    analysis = json.loads(stock_analysis(ticker))

    # 투자 가치 판단을 위한 기준
    revenue_growth = analysis["성장성"]["매출 성장률"]
    roe = analysis["수익성"]["ROE"]
    per = analysis["밸류에이션"]["PER"]
    dividend_yield = analysis["배당"]["배당 수익률"]

    # 평가 메시지 작성
    evaluation_prompt = f"""
    다음은 주식 {ticker}의 주요 재무 지표입니다:
    - 매출 성장률: {revenue_growth}
    - ROE: {roe}
    - PER: {per}
    - 배당 수익률: {dividend_yield}

    이 주식의 투자 가치를 평가하고, 투자 여부를 결정해주세요. 
    간략한 이유도 포함하여 설명하세요.
    """
    llm = ChatOpenAI(model = "gpt-4o-mini",temperature=0)
    decision = llm.invoke(evaluation_prompt)
    
    
    return decision


# 투자 가치 판단
decision = investment_decision("AAPL")
print("투자 가치 평가 결과:")
print(decision)

투자 가치 평가 결과:
content='AAPL(Apple Inc.)의 주요 재무 지표를 바탕으로 투자 가치를 평가해보겠습니다.\n\n1. **매출 성장률 (2.02%)**: 매출 성장률이 낮은 편입니다. 이는 AAPL이 이미 성숙기에 접어들었거나 시장에서의 성장 기회가 제한적일 수 있음을 나타냅니다. 그러나 안정적인 매출을 유지하고 있다는 점은 긍정적입니다.\n\n2. **ROE (164.59%)**: ROE(자기자본이익률)가 매우 높습니다. 이는 AAPL이 주주에게 높은 수익을 제공하고 있다는 것을 의미합니다. 높은 ROE는 기업의 효율성과 수익성을 나타내는 중요한 지표입니다.\n\n3. **PER (32.59)**: PER(주가수익비율)가 상대적으로 높은 편입니다. 이는 시장에서 AAPL의 미래 성장 가능성을 높게 평가하고 있다는 것을 의미하지만, 동시에 주가가 고평가되어 있을 가능성도 있습니다.\n\n4. **배당 수익률 (53.00%)**: 배당 수익률이 매우 높습니다. 이는 AAPL이 주주에게 상당한 배당금을 지급하고 있다는 것을 나타내며, 안정적인 현금 흐름을 가지고 있다는 신호일 수 있습니다. 그러나 이 수치가 비정상적으로 높다면, 지속 가능성에 대한 우려가 있을 수 있습니다.\n\n### 결론:\nAAPL은 높은 ROE와 배당 수익률을 가지고 있어 투자 매력이 있지만, 낮은 매출 성장률과 높은 PER은 주의가 필요합니다. 안정적인 수익을 추구하는 투자자에게는 매력적일 수 있지만, 성장성을 중시하는 투자자에게는 다소 아쉬운 점이 있을 수 있습니다. \n\n따라서, AAPL에 대한 투자는 **신중하게 고려해야 하며**, 안정적인 배당과 높은 수익성을 중시하는 투자자에게는 긍정적일 수 있지만, 성장 가능성을 중시하는 투자자에게는 적합하지 않을 수 있습니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 443, 'prompt_tok

### Supervisior 에이전트 정의

### Supervisior 에이전트 정의

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder